In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

print("DeliveryRisk Engine - Baseline Feature Engineering")

DeliveryRisk Engine - Baseline Feature Engineering


In [2]:
DATA_DIR = Path("../data/raw")

orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
order_items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(DATA_DIR / "olist_sellers_dataset.csv")

print("Data loaded.")

Data loaded.


In [3]:
timestamp_cols = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in timestamp_cols:
    orders[col] = pd.to_datetime(orders[col])

delivered_orders = orders[
    orders["order_status"] == "delivered"
].copy()

delivered_orders["late_delivery"] = (
    delivered_orders["order_delivered_customer_date"]
    > delivered_orders["order_estimated_delivery_date"]
).astype(int)

print("Delivered orders:", len(delivered_orders))
print(delivered_orders["late_delivery"].value_counts())

Delivered orders: 96478
late_delivery
0    88652
1     7826
Name: count, dtype: int64


In [4]:
order_features = delivered_orders[
    [
        "order_id",
        "customer_id",
        "order_purchase_timestamp",
        "late_delivery"
    ]
].copy()

order_features["purchase_hour"] = (
    order_features["order_purchase_timestamp"].dt.hour
)

order_features["purchase_dayofweek"] = (
    order_features["order_purchase_timestamp"].dt.dayofweek
)

order_features["purchase_month"] = (
    order_features["order_purchase_timestamp"].dt.month
)

order_features["is_weekend"] = (
    order_features["purchase_dayofweek"] >= 5
).astype(int)

order_features.head()

,order_id,customer_id,order_purchase_timestamp,late_delivery,purchase_hour,purchase_dayofweek,purchase_month,is_weekend
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02 10:56:33,0,10,0,10,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24 20:41:37,0,20,1,7,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08 08:38:49,0,8,2,8,0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-11-18 19:28:06,0,19,5,11,1
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-13 21:18:39,0,21,1,2,0


In [6]:
customer_features = customers[
    [
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ]
].copy()

order_features = order_features.merge(
    customer_features,
    on="customer_id",
    how="left"
)

print(order_features.shape)

(96478, 16)


In [9]:
item_features = (
    order_items
    .groupby("order_id")
    .agg(
        item_count=("order_item_id", "count"),
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum"),
        avg_item_price=("price", "mean"),
        unique_product_count=("product_id", "nunique"),
        unique_seller_count=("seller_id", "nunique")
    )
    .reset_index()
)

item_features.head()

,order_id,item_count,total_price,total_freight,avg_item_price,unique_product_count,unique_seller_count
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,58.90,1,1
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,239.90,1,1
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,199.00,1,1
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,12.99,1,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,199.90,1,1


In [10]:
order_features = order_features.merge(
    item_features,
    on="order_id",
    how="left"
)

print(order_features.shape)

(96478, 26)


In [11]:
item_seller = order_items[
    ["order_id", "seller_id"]
].merge(
    sellers[
        [
            "seller_id",
            "seller_zip_code_prefix",
            "seller_city",
            "seller_state"
        ]
    ],
    on="seller_id",
    how="left"
)

seller_features = (
    item_seller
    .groupby("order_id")
    .agg(
        seller_count=("seller_id", "nunique"),
        seller_state_count=("seller_state", "nunique"),
        avg_seller_zip=("seller_zip_code_prefix", "mean")
    )
    .reset_index()
)

order_features = order_features.merge(
    seller_features,
    on="order_id",
    how="left"
)

print(order_features.shape)

(96478, 29)


In [12]:
print("Rows:", len(order_features))
print("Unique orders:", order_features["order_id"].nunique())
print("Columns:", len(order_features.columns))

Rows: 96478
Unique orders: 96478
Columns: 29


In [13]:
OUTPUT_PATH = Path("../data/processed/baseline_features.csv")

order_features.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Saved to: {OUTPUT_PATH}")

Saved to: ..\data\processed\baseline_features.csv


In [14]:
item_product = order_items[
    ["order_id", "product_id"]
].merge(
    products[
        [
            "product_id",
            "product_category_name",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ],
    on="product_id",
    how="left"
)

product_features = (
    item_product
    .groupby("order_id")
    .agg(
        total_product_weight_g=("product_weight_g", "sum"),
        avg_product_weight_g=("product_weight_g", "mean"),
        avg_product_length_cm=("product_length_cm", "mean"),
        avg_product_height_cm=("product_height_cm", "mean"),
        avg_product_width_cm=("product_width_cm", "mean"),
        unique_product_categories=("product_category_name", "nunique")
    )
    .reset_index()
)

print("Product features created:", product_features.shape)

Product features created: (98666, 7)


In [15]:
order_features = order_features.merge(
    product_features,
    on="order_id",
    how="left"
)

print("Shape:", order_features.shape)
print("Unique orders:", order_features["order_id"].nunique())

Shape: (96478, 35)
Unique orders: 96478


In [16]:
item_seller = order_items[
    ["order_id", "seller_id"]
].merge(
    sellers[
        [
            "seller_id",
            "seller_zip_code_prefix",
            "seller_city",
            "seller_state"
        ]
    ],
    on="seller_id",
    how="left"
)

seller_features = (
    item_seller
    .groupby("order_id")
    .agg(
        seller_count=("seller_id", "nunique"),
        seller_state_count=("seller_state", "nunique"),
        primary_seller_state=(
            "seller_state",
            lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
        )
    )
    .reset_index()
)

print("Seller features:", seller_features.shape)

Seller features: (98666, 4)


In [17]:
order_features = order_features.merge(
    seller_features,
    on="order_id",
    how="left"
)

print("Shape:", order_features.shape)
print("Unique orders:", order_features["order_id"].nunique())

Shape: (96478, 38)
Unique orders: 96478


In [18]:
shipping = order_items[
    [
        "order_id",
        "shipping_limit_date"
    ]
].copy()

shipping["shipping_limit_date"] = pd.to_datetime(
    shipping["shipping_limit_date"]
)

shipping = shipping.merge(
    order_features[
        [
            "order_id",
            "order_purchase_timestamp"
        ]
    ],
    on="order_id",
    how="inner"
)

shipping["shipping_window_hours"] = (
    shipping["shipping_limit_date"]
    - shipping["order_purchase_timestamp"]
).dt.total_seconds() / 3600

shipping_features = (
    shipping
    .groupby("order_id")
    .agg(
        avg_shipping_window_hours=(
            "shipping_window_hours",
            "mean"
        ),
        min_shipping_window_hours=(
            "shipping_window_hours",
            "min"
        )
    )
    .reset_index()
)

print("Shipping features:", shipping_features.shape)

Shipping features: (96478, 3)


In [19]:
order_features = order_features.merge(
    shipping_features,
    on="order_id",
    how="left"
)

print("Shape:", order_features.shape)
print("Unique orders:", order_features["order_id"].nunique())

Shape: (96478, 40)
Unique orders: 96478


In [20]:
order_features.columns.tolist()

['order_id',
 'customer_id',
 'order_purchase_timestamp',
 'late_delivery',
 'purchase_hour',
 'purchase_dayofweek',
 'purchase_month',
 'is_weekend',
 'customer_unique_id_x',
 'customer_zip_code_prefix_x',
 'customer_city_x',
 'customer_state_x',
 'customer_unique_id_y',
 'customer_zip_code_prefix_y',
 'customer_city_y',
 'customer_state_y',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'item_count',
 'total_price',
 'total_freight',
 'avg_item_price',
 'unique_product_count',
 'unique_seller_count',
 'seller_count_x',
 'seller_state_count_x',
 'avg_seller_zip',
 'total_product_weight_g',
 'avg_product_weight_g',
 'avg_product_length_cm',
 'avg_product_height_cm',
 'avg_product_width_cm',
 'unique_product_categories',
 'seller_count_y',
 'seller_state_count_y',
 'primary_seller_state',
 'avg_shipping_window_hours',
 'min_shipping_window_hours']

In [21]:
order_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 96478 entries, 0 to 96477
Data columns (total 40 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   order_id                    96478 non-null  str           
 1   customer_id                 96478 non-null  str           
 2   order_purchase_timestamp    96478 non-null  datetime64[us]
 3   late_delivery               96478 non-null  int64         
 4   purchase_hour               96478 non-null  int32         
 5   purchase_dayofweek          96478 non-null  int32         
 6   purchase_month              96478 non-null  int32         
 7   is_weekend                  96478 non-null  int64         
 8   customer_unique_id_x        96478 non-null  str           
 9   customer_zip_code_prefix_x  96478 non-null  int64         
 10  customer_city_x             96478 non-null  str           
 11  customer_state_x            96478 non-null  str           
 12  c

In [22]:
order_features.isnull().sum().sort_values(ascending=False)

avg_product_weight_g          16
avg_product_length_cm         16
avg_product_width_cm          16
avg_product_height_cm         16
order_id                       0
customer_id                    0
order_purchase_timestamp       0
late_delivery                  0
is_weekend                     0
purchase_month                 0
purchase_dayofweek             0
purchase_hour                  0
customer_unique_id_x           0
customer_zip_code_prefix_x     0
customer_city_x                0
customer_state_x               0
customer_unique_id             0
customer_zip_code_prefix       0
customer_city                  0
customer_state                 0
customer_unique_id_y           0
customer_zip_code_prefix_y     0
customer_city_y                0
customer_state_y               0
avg_item_price                 0
total_freight                  0
total_price                    0
item_count                     0
seller_state_count_x           0
seller_count_x                 0
unique_pro

In [23]:
order_features["late_delivery"].value_counts()

late_delivery
0    88652
1     7826
Name: count, dtype: int64

In [24]:
order_features["late_delivery"].value_counts(normalize=True) * 100

late_delivery
0    91.888306
1     8.111694
Name: proportion, dtype: float64

In [25]:
id_columns = [
    "order_id",
    "customer_id",
    "customer_unique_id"
]

model_data = order_features.drop(
    columns=id_columns
).copy()

print("Model data shape:", model_data.shape)

Model data shape: (96478, 37)


In [26]:
categorical_columns = model_data.select_dtypes(
    include=["object"]
).columns.tolist()

print(categorical_columns)

['customer_unique_id_x', 'customer_city_x', 'customer_state_x', 'customer_unique_id_y', 'customer_city_y', 'customer_state_y', 'customer_city', 'customer_state', 'primary_seller_state']


C:\Users\anant\AppData\Local\Temp\ipykernel_34108\3258765731.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = model_data.select_dtypes(


In [27]:
numerical_columns = model_data.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print(numerical_columns)

['late_delivery', 'is_weekend', 'customer_zip_code_prefix_x', 'customer_zip_code_prefix_y', 'customer_zip_code_prefix', 'item_count', 'total_price', 'total_freight', 'avg_item_price', 'unique_product_count', 'unique_seller_count', 'seller_count_x', 'seller_state_count_x', 'avg_seller_zip', 'total_product_weight_g', 'avg_product_weight_g', 'avg_product_length_cm', 'avg_product_height_cm', 'avg_product_width_cm', 'unique_product_categories', 'seller_count_y', 'seller_state_count_y', 'avg_shipping_window_hours', 'min_shipping_window_hours']


In [33]:
model_data = model_data.drop(
    columns=["avg_seller_zip"],
    errors="ignore"
)

In [34]:
print(model_data.dtypes)

late_delivery                   int64
purchase_hour                   int32
purchase_dayofweek              int32
purchase_month                  int32
is_weekend                      int64
customer_unique_id_x              str
customer_zip_code_prefix_x      int64
customer_city_x                   str
customer_state_x                  str
customer_unique_id_y              str
customer_zip_code_prefix_y      int64
customer_city_y                   str
customer_state_y                  str
customer_zip_code_prefix        int64
customer_city                     str
customer_state                    str
item_count                      int64
total_price                   float64
total_freight                 float64
avg_item_price                float64
unique_product_count            int64
unique_seller_count             int64
seller_count_x                  int64
seller_state_count_x            int64
total_product_weight_g        float64
avg_product_weight_g          float64
avg_product_

In [35]:
print("Rows:", len(model_data))
print("Columns:", len(model_data.columns))

Rows: 96478
Columns: 35


In [38]:
model_data = order_features.copy()

# Sort chronologically BEFORE removing the timestamp
model_data = model_data.sort_values(
    "order_purchase_timestamp"
).reset_index(drop=True)

# Remove identifiers and fields that should not go directly into the model
model_data = model_data.drop(
    columns=[
        "order_id",
        "customer_id",
        "customer_unique_id",
        "order_purchase_timestamp",
        "avg_seller_zip"
    ],
    errors="ignore"
)

print("Rows:", len(model_data))
print("Columns:", len(model_data.columns))
print("\nRemaining columns:")
print(model_data.columns.tolist())

Rows: 96478
Columns: 35

Remaining columns:
['late_delivery', 'purchase_hour', 'purchase_dayofweek', 'purchase_month', 'is_weekend', 'customer_unique_id_x', 'customer_zip_code_prefix_x', 'customer_city_x', 'customer_state_x', 'customer_unique_id_y', 'customer_zip_code_prefix_y', 'customer_city_y', 'customer_state_y', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'item_count', 'total_price', 'total_freight', 'avg_item_price', 'unique_product_count', 'unique_seller_count', 'seller_count_x', 'seller_state_count_x', 'total_product_weight_g', 'avg_product_weight_g', 'avg_product_length_cm', 'avg_product_height_cm', 'avg_product_width_cm', 'unique_product_categories', 'seller_count_y', 'seller_state_count_y', 'primary_seller_state', 'avg_shipping_window_hours', 'min_shipping_window_hours']


In [39]:
model_data.isnull().sum().sort_values(ascending=False)

avg_product_length_cm         16
avg_product_width_cm          16
avg_product_height_cm         16
avg_product_weight_g          16
purchase_dayofweek             0
late_delivery                  0
purchase_hour                  0
purchase_month                 0
is_weekend                     0
customer_unique_id_x           0
customer_zip_code_prefix_x     0
customer_city_y                0
customer_state_y               0
customer_zip_code_prefix       0
customer_city                  0
customer_city_x                0
customer_state_x               0
customer_unique_id_y           0
customer_zip_code_prefix_y     0
total_freight                  0
total_price                    0
item_count                     0
customer_state                 0
seller_count_x                 0
unique_seller_count            0
avg_item_price                 0
unique_product_count           0
seller_state_count_x           0
total_product_weight_g         0
unique_product_categories      0
seller_cou

In [40]:
for i, col in enumerate(model_data.columns):
    print(i, col)

0 late_delivery
1 purchase_hour
2 purchase_dayofweek
3 purchase_month
4 is_weekend
5 customer_unique_id_x
6 customer_zip_code_prefix_x
7 customer_city_x
8 customer_state_x
9 customer_unique_id_y
10 customer_zip_code_prefix_y
11 customer_city_y
12 customer_state_y
13 customer_zip_code_prefix
14 customer_city
15 customer_state
16 item_count
17 total_price
18 total_freight
19 avg_item_price
20 unique_product_count
21 unique_seller_count
22 seller_count_x
23 seller_state_count_x
24 total_product_weight_g
25 avg_product_weight_g
26 avg_product_length_cm
27 avg_product_height_cm
28 avg_product_width_cm
29 unique_product_categories
30 seller_count_y
31 seller_state_count_y
32 primary_seller_state
33 avg_shipping_window_hours
34 min_shipping_window_hours


In [41]:
# ============================================================
# CLEAN BASELINE FEATURE SET
# ============================================================

baseline_columns = [
    # Target
    "late_delivery",

    # Time
    "purchase_hour",
    "purchase_dayofweek",
    "purchase_month",
    "is_weekend",

    # Customer
    "customer_state",

    # Order / item features
    "item_count",
    "total_price",
    "total_freight",
    "avg_item_price",
    "unique_product_count",
    "unique_seller_count",

    # Product features
    "total_product_weight_g",
    "avg_product_weight_g",
    "avg_product_length_cm",
    "avg_product_height_cm",
    "avg_product_width_cm",
    "unique_product_categories",

    # Seller features
    "seller_count_x",
    "seller_state_count_x",
    "primary_seller_state",

    # Shipping
    "avg_shipping_window_hours",
    "min_shipping_window_hours",
]

# Check that every requested column exists
missing_columns = [
    col for col in baseline_columns
    if col not in order_features.columns
]

print("Missing requested columns:", missing_columns)

# Create clean dataset
baseline_data = order_features[baseline_columns].copy()

print("Baseline shape:", baseline_data.shape)
print("Unique rows:", len(baseline_data))
print("\nColumns:")
print(baseline_data.columns.tolist())

Missing requested columns: []
Baseline shape: (96478, 23)
Unique rows: 96478

Columns:
['late_delivery', 'purchase_hour', 'purchase_dayofweek', 'purchase_month', 'is_weekend', 'customer_state', 'item_count', 'total_price', 'total_freight', 'avg_item_price', 'unique_product_count', 'unique_seller_count', 'total_product_weight_g', 'avg_product_weight_g', 'avg_product_length_cm', 'avg_product_height_cm', 'avg_product_width_cm', 'unique_product_categories', 'seller_count_x', 'seller_state_count_x', 'primary_seller_state', 'avg_shipping_window_hours', 'min_shipping_window_hours']


In [42]:
baseline_data.isnull().sum().sort_values(ascending=False)

avg_product_width_cm         16
avg_product_length_cm        16
avg_product_weight_g         16
avg_product_height_cm        16
late_delivery                 0
purchase_hour                 0
purchase_dayofweek            0
item_count                    0
customer_state                0
is_weekend                    0
purchase_month                0
unique_product_count          0
avg_item_price                0
total_price                   0
total_freight                 0
unique_seller_count           0
total_product_weight_g        0
unique_product_categories     0
seller_count_x                0
seller_state_count_x          0
primary_seller_state          0
avg_shipping_window_hours     0
min_shipping_window_hours     0
dtype: int64